# Predictive Circuit Coding Training

This notebook uses the shared Drive dataset as a read-only source, keeps artifacts local during the run for reliability, and exports the finished run to a clearly separate Drive folder when training is done.

Default behavior is simple: train on a small notebook-only familiar-session subset so debug runs stay fast and coherent.

In [ ]:
# Configuration - edit these values, then Run All

# Dataset scope
USE_FULL_DATASET = False  # True = full dataset (slow); False = notebook subset
EXPERIENCE_LEVEL = 'Familiar'
MAX_SESSIONS = 50

# Split fractions
TRAIN_FRACTION = 0.6
VALID_FRACTION = 0.1
DISCOVERY_FRACTION = 0.2
TEST_FRACTION = 0.1

# Training budget suggestions
# Debug / light subset: NUM_EPOCHS = 8-10, TRAIN_STEPS_PER_EPOCH = 128, VALIDATION_STEPS = 16
# Medium run (~25-50 sessions): NUM_EPOCHS = 20, TRAIN_STEPS_PER_EPOCH = 256, VALIDATION_STEPS = 32-64
# Deep A100 run: NUM_EPOCHS = 40-50, TRAIN_STEPS_PER_EPOCH = 512+, VALIDATION_STEPS = 64-128
NUM_EPOCHS = 45
TRAIN_STEPS_PER_EPOCH = 512
VALIDATION_STEPS = 64

# Notebook output cadence
NOTEBOOK_STEP_LOG_EVERY = 100  # print a log line every N optimizer steps


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'

# Shared folder from the storage-heavy Drive account: read dataset from here.
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'

# Keep exported Colab outputs in a clearly different folder name so they do not get
# confused with the source dataset bundle.
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
%cd {repo_root}
# Keep Colab installs minimal to avoid restart prompts from preloaded widget packages.
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if repo_artifacts_root.exists():
    shutil.rmtree(repo_artifacts_root)
repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
from predictive_circuit_coding.utils import (
    NotebookDatasetConfig,
    NotebookTrainingConfig,
    NotebookStageReporter,
    export_notebook_training_artifacts,
    load_notebook_split_counts,
    prepare_notebook_runtime_context,
    resolve_notebook_checkpoint,
    run_streaming_command,
    verify_paths_exist,
)

NOTEBOOK_DATASET = NotebookDatasetConfig(
    use_full_dataset=USE_FULL_DATASET,
    experience_level=EXPERIENCE_LEVEL,
    max_sessions=MAX_SESSIONS,
    train_fraction=TRAIN_FRACTION,
    valid_fraction=VALID_FRACTION,
    discovery_fraction=DISCOVERY_FRACTION,
    test_fraction=TEST_FRACTION,
)

NOTEBOOK_TRAINING = NotebookTrainingConfig(
    num_epochs=NUM_EPOCHS,
    train_steps_per_epoch=TRAIN_STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
)

reporter = NotebookStageReporter(name='colab-train', expected_duration='2-10 minutes for debug runs, longer for full A100 runs')
reporter.banner('Predictive Circuit Coding Training', subtitle='Setup, preflight, runtime selection, training, evaluation, and export')

base_experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
runtime_context = prepare_notebook_runtime_context(
    base_experiment_config=base_experiment_config,
    runtime_experiment_config=repo_root / 'colab_runtime_experiment.yaml',
    session_catalog_csv=repo_root / 'data' / 'allen_visual_behavior_neuropixels' / 'manifests' / 'session_catalog.csv',
    artifact_root=repo_root / 'artifacts',
    dataset_config=NOTEBOOK_DATASET,
    training_config=NOTEBOOK_TRAINING,
    step_log_every=NOTEBOOK_STEP_LOG_EVERY,
)
experiment_config = runtime_context.experiment_config_path
checkpoint_dir = runtime_context.checkpoint_dir
summary_path = runtime_context.summary_path
resume_checkpoint = runtime_context.checkpoint_path
runtime_split_manifest_path = runtime_context.runtime_split_manifest_path
drive_run_export_root = drive_export_root / runtime_context.run_id / 'run_1' / 'train'

print('Run ID:', runtime_context.run_id)
print('Experiment config:', experiment_config)
print('Data config:', data_config)
print('Runtime split manifest:', runtime_split_manifest_path)
if not NOTEBOOK_DATASET.use_full_dataset:
    print('Target experience level:', NOTEBOOK_DATASET.experience_level)
    print('Max sessions:', NOTEBOOK_DATASET.max_sessions)
    print('Requested split fractions:', {
        'train': NOTEBOOK_DATASET.train_fraction,
        'valid': NOTEBOOK_DATASET.valid_fraction,
        'discovery': NOTEBOOK_DATASET.discovery_fraction,
        'test': NOTEBOOK_DATASET.test_fraction,
    })
    print('Selected sessions:', runtime_context.selected_session_count)
print('Notebook step-log cadence:', NOTEBOOK_STEP_LOG_EVERY)
print('Training budget:', {
    'num_epochs': NOTEBOOK_TRAINING.num_epochs,
    'train_steps_per_epoch': NOTEBOOK_TRAINING.train_steps_per_epoch,
    'validation_steps': NOTEBOOK_TRAINING.validation_steps,
})
print('Runtime config dir:', runtime_context.runtime_config_dir)
print('Resolved checkpoint dir:', checkpoint_dir.resolve())
print('Resolved summary path:', summary_path.resolve())
print('Checkpoint path:', resume_checkpoint)
print('Notebook profile:', runtime_context.profile_path)
print('Drive export path:', drive_run_export_root)

In [ ]:
reporter.begin('preflight', next_artifact='best checkpoint + evaluation summary')
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing config files or dataset bundle for training notebook.'

split_manifest_for_counts = runtime_split_manifest_path if runtime_split_manifest_path.exists() else (repo_root / 'data' / 'allen_visual_behavior_neuropixels' / 'splits' / 'split_manifest.json')
realized_split_counts = load_notebook_split_counts(
    split_manifest_path=split_manifest_for_counts,
)
print('Realized split counts:', realized_split_counts)

reporter.finish('preflight')

In [ ]:
reporter.begin('training', next_artifact='local checkpoint under artifacts/checkpoints')
%cd {repo_root}
run_streaming_command([
    'pcc-train',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
resume_checkpoint = resolve_notebook_checkpoint(summary_path=summary_path, checkpoint_dir=checkpoint_dir)
if not resume_checkpoint.exists():
    checkpoint_candidates = sorted(checkpoint_dir.glob('*.pt'))
    print('Available checkpoints:', [path.name for path in checkpoint_candidates])
    raise AssertionError(f'Expected checkpoint was not written: {resume_checkpoint}')
reporter.note_checkpoint(resume_checkpoint)
reporter.finish('training')

In [ ]:
reporter.begin('evaluation', next_artifact='local evaluation summary json')
%cd {repo_root}
run_streaming_command([
    'pcc-evaluate',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(resume_checkpoint),
    '--split', 'valid',
], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
reporter.finish('evaluation')

In [ ]:
reporter.begin('export', next_artifact='Drive copy of local artifacts')
drive_run_export_root = export_notebook_training_artifacts(
    drive_export_root=drive_export_root,
    local_artifact_root=repo_artifacts_root,
    run_id=runtime_context.run_id,
)
print('Exported local artifacts to:', drive_run_export_root)
reporter.finish('export')

## Results

Visualization and summary of the completed run.


In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

with open(summary_path, 'r', encoding='utf-8') as fh:
    train_summary = json.load(fh)
train_metrics = train_summary.get('metrics', {})

# Find validation evaluation JSON beside the checkpoint.
eval_data = None
candidates = list(checkpoint_dir.glob('*_valid_evaluation.json'))
if not candidates:
    candidates = list(checkpoint_dir.parent.glob('*_valid_evaluation.json'))
if candidates:
    with open(sorted(candidates)[-1], 'r', encoding='utf-8') as fh:
        eval_data = json.load(fh)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Model MSE vs baseline MSE
source = eval_data.get('metrics', {}) if eval_data else train_metrics
title_suffix = '(validation split)' if eval_data else '(train metrics — eval not found)'
raw = source.get('predictive_raw_mse', float('nan'))
baseline = source.get('predictive_baseline_mse', float('nan'))
bars = axes[0].bar(['Model\n(raw MSE)', 'Baseline\n(prev-patch)'], [raw, baseline],
                   color=['#4c72b0', '#dd8452'], width=0.5)
axes[0].set_title(f'Model vs Baseline MSE {title_suffix}')
axes[0].set_ylabel('MSE')
axes[0].set_ylim(bottom=0)
for bar, v in zip(bars, [raw, baseline]):
    if v == v:  # skip NaN
        axes[0].text(bar.get_x() + bar.get_width() / 2, v * 1.02, f'{v:.4f}', ha='center', fontsize=9)

# Panel 2: Loss breakdown
losses = train_summary.get('losses', {})
loss_vals = [losses.get(k, float('nan')) for k in ('predictive_loss', 'reconstruction_loss', 'total_loss')]
bars2 = axes[1].bar(['Predictive', 'Reconstruction', 'Total'], loss_vals,
                    color=['#55a868', '#c44e52', '#8172b2'], width=0.5)
axes[1].set_title('Final Training Loss Breakdown')
axes[1].set_ylabel('Loss')
axes[1].set_ylim(bottom=0)
for bar, v in zip(bars2, loss_vals):
    if v == v:
        axes[1].text(bar.get_x() + bar.get_width() / 2, v * 1.02, f'{v:.4f}', ha='center', fontsize=9)

fig.suptitle('Training Run Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import json

with open(summary_path, 'r', encoding='utf-8') as fh:
    ts = json.load(fh)

m = ts.get('metrics', {})
imp = m.get('predictive_improvement')
raw = m.get('predictive_raw_mse')
baseline = m.get('predictive_baseline_mse')
losses = ts.get('losses', {})

if imp is not None:
    direction = 'better than' if imp > 0 else 'worse than'
    imp_str = f'{abs(imp * 100):.1f}% {direction} the previous-patch baseline'
elif raw and baseline and baseline > 0:
    pct = (1 - raw / baseline) * 100
    direction = 'better than' if pct > 0 else 'worse than'
    imp_str = f'{abs(pct):.1f}% {direction} the previous-patch baseline'
else:
    imp_str = 'comparison to baseline not available'

session_label = (
    f'{NOTEBOOK_DATASET.max_sessions}-session {NOTEBOOK_DATASET.experience_level} subset'
    if not NOTEBOOK_DATASET.use_full_dataset else 'the full dataset'
)
print(
    f'Training completed on {session_label} ({ts.get("dataset_id", "?")}).\n'
    f'Best checkpoint: epoch {ts.get("best_epoch", "?")} of {ts.get("epoch", "?")} total.\n'
    f'The model was {imp_str}.\n'
    f'Final losses — total: {losses.get("total_loss", float("nan")):.4f}, '
    f'predictive: {losses.get("predictive_loss", float("nan")):.4f}, '
    f'reconstruction: {losses.get("reconstruction_loss", float("nan")):.4f}.'
)